## Aprendizagem Automática (APRAU) - Grupo 01
#### Mestrado em Engenharia Informática
##### First Assignment

- Daniel Braga - 1200801
- Diogo Martins - 1211978
- Gaspar Vieira - 1251542

# 1.Análise Exploratória de Dados (Exploratory Data Analysis, EDA)

## 1.1. Estatisticas Descritivas

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")

df = pd.read_csv("dataset_grupo_01.csv")
df_original = df.copy() 
TARGET_C, TARGET_R = "target_class", "target_regression" 


print(f"Dimensão inicial: {df.shape}")
print(f"Linhas duplicadas: {df.duplicated().sum()}")
print(f"Valores em falta: {df.isna().sum().sum()}")

display(df.describe().T)

Dimensão inicial: (2000, 35)
Linhas duplicadas: 0
Valores em falta: 0


,count,mean,std,min,25%,50%,75%,max
age,2000.0,49.817500,21.924086,2.000000,31.000000,50.000000,68.000000,89.000000
age_normalized,2000.0,0.549626,0.252001,0.000000,0.333333,0.551724,0.758620,1.000000
bmi,2000.0,27.360674,6.883579,13.750000,22.200000,27.320000,32.402500,68.530000
HbA1c_level,2000.0,5.823790,0.928032,3.500000,5.374129,5.494662,6.750000,9.000000
cholesterol,2000.0,223.647108,34.915456,150.000000,201.000000,224.842938,242.000000,299.000000
sleep_hours,2000.0,6.941153,1.415719,4.000000,6.200000,6.982952,7.700000,10.000000
triglycerides,2000.0,178.811071,66.089529,50.000000,131.750000,179.000000,243.026316,388.000000
blood_pressure,2000.0,149.687674,3.626313,121.000000,148.625000,149.587719,150.654054,176.000000
crp_level,2000.0,7.483729,0.882769,0.467803,7.294863,7.458483,7.767442,14.922016
homocysteine_level,2000.0,12.453406,0.844962,5.309536,12.287763,12.504625,12.681166,19.959139


In [2]:
# Tabelas de frequência para variáveis categóricas
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for col in categorical_cols:
    print(f"\n{col} — frequências")
    display(df[col].value_counts(dropna=False).to_frame("frequência")
            .assign(percentagem=lambda x: (x["frequência"] / len(df) * 100).round(2)))

NameError: name 'np' is not defined

## 1.2. Análise Univariada

In [ ]:
numeric_predictors = [c for c in numeric_cols if c not in [TARGET_R]]

for col in numeric_predictors:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    sns.histplot(data=df, x=col, kde=True, ax=axes[0])
    axes[0].set_title(f"Distribuição de {col}")
    sns.boxplot(data=df, x=col, ax=axes[1])
    axes[1].set_title(f"Boxplot de {col}")
    plt.tight_layout()
    plt.show()


fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.histplot(data=df, x=TARGET_R, kde=True, ax=axes[0])
axes[0].set_title(f"Distribuição de {TARGET_R}")
sns.boxplot(data=df, x=TARGET_R, ax=axes[1])
axes[1].set_title(f"Boxplot de {TARGET_R}")
plt.tight_layout()
plt.show()

for col1, col2 in [
    ('systolic_bp', 'systolic_index'),
    ('ldl', 'ldl_alternative'),
]:
    id_count = (df[col1] == df[col2]).sum()
    total_rows = len(df)
    pct_identical = (id_count / total_rows) * 100

    print(
        f'{col1} vs. {col2}: '
        f'{id_count} de {total_rows} linhas idênticas ({pct_identical:.2f}%)'
    )

Desta primeira análise conseguimos concluir que o calcium_baseline, tal como o nome nome indica, é uma variável constante. Adicionalmente, o systolic_index e systolic_bp apresentam valores indenticos. O ldl e ldl_alternitive aparentam ser parecidos porém, os seus valores são totalmente diferentes. Observamos também que os atributos age e age_normalized são possivelmente idênticas, sendo redundante manter as duas, o que iremos analisar posteriormente.

In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(8, 3.5))
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order)
    plt.title(f"Distribuição de {col}")
    plt.xlabel(col)
    plt.ylabel("Número de registos")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

results = []

for col in categorical_cols:
    if col not in df_original.columns:
        continue

    counts = df_original[col].value_counts()
    if counts.empty:
        continue

    results.append({
        "Column": col,
        "Most common": counts.index[0],
        "Least common": counts.index[-1],
        "Most common count": counts.iloc[0],
        "Least common count": counts.iloc[-1],
        "Majority/minority ratio": round(counts.iloc[0] / counts.iloc[-1], 2),
    })

display(pd.DataFrame(results))

A análise da distribuição das variáveis permite concluir que existem diferentes níveis de desequilíbrio entre as categorias. Destaca-se a variável low_hdl_cholesterol, com uma razão de 4,35, seguida de bmi_level (2,84) e age_level (2,23), indicando uma distribuição menos equilibrada nestas variáveis. Em contrapartida, variáveis como gender, high_hdl_cholesterol e employment_status apresentam distribuições mais equilibradas entre as categorias.

## 1.3. Qualidade dos Dados 

In [ ]:

corr_matrix = df[numeric_predictors].corr()
plt.figure(figsize=(20,17))
sns.heatmap(corr_matrix, vmin=-1, vmax=1, annot=True, fmt='.2f', center=1) 
plt.show()

Com o recurso a este _heatmap_ e os restantes gráficos, confirma-mos que tanto o calcium_baseline, como age_normalize e o systolic_index são tabelas que podemos remover para treinar o nosso modelo. Uma vez que ldl e ldl_alternative não têm uma correlação alta, não podemos removelos como pensamos anteriormente. 

Assim, removemos os seguintes atributos:
- **age_normalized**: uma vez que é igual a age;
- **calcium_baseline**: uma vez que é um valor constante, não nos interessa;
- **systolic_index**: uma vez que é igual ao systolic_bp.
- **age_level** e **bmi_level**: uma vez que são atributos categóricos cujos valores de age e bmi estão, respetivamente, agregados em Young, Middle, Mature, Senior e Underweigh, Normal, Overweight e Obese. Assim, age_level e bmi_level, representam, com menos qualidade, os atributos age e bmi. 



Adicionalmente, queremos explorar de forma mais visível a correlação entre os _targets_ e as restantes atributos.

## 1.4. Relação entre atributos e `target_class`

In [ ]:
class_counts = df[TARGET_C].value_counts()
class_percent = df[TARGET_C].value_counts(normalize=True) * 100

class_distribution = pd.DataFrame({
    "frequência": class_counts,
    "percentagem (%)": class_percent.round(2)
})
display(class_distribution)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_C, order=class_counts.index)
plt.title("Distribuição das classes")
plt.xlabel("Classe")
plt.ylabel("Número de registos")
plt.tight_layout()
plt.show()

In [ ]:
for col in df.select_dtypes(include=['object', 'category']).columns:
    unique_vals = set(df[col].dropna().unique())
    if unique_vals.issubset({'Yes', 'No'}):
        df[col] = df[col].map({'Yes': 1, 'No': 0})

ordinal_mappings = {
    'age_level': {'Young': 0, 'Middle': 1, 'Mature': 2, 'Senior': 3},
    'bmi_level': {'Underweight': 0, 'Normal': 1, 'Overweight': 2, 'Obese': 3},
    'stress_level': {'Low': 0, 'Medium': 1, 'High': 2},
    'physical_activity': {'Low': 0, 'Moderate': 1, 'High': 2}
}

for col, mapping in ordinal_mappings.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)
        
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

bool_cols = df.select_dtypes(include=['bool', 'uint8']).columns
df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
df['target_class_DI_HY'] = (~df['target_class_HY']) & (~df['target_class_N'])

target_class_cols = ['target_class_HY', 'target_class_N', 'target_class_DI_HY']

df_graph = df.copy().drop(['target_class_HY', 'target_class_N',], axis=1)

corr_data = df_graph.select_dtypes(include="number").copy()
for col in target_class_cols:
    if col in df.columns:
        corr_data[col] = df[col]

corr_matrix = corr_data.corr()

for t_col in target_class_cols:
    if t_col in corr_matrix.columns:
        print(f"\n--- Correlação com a Classe: {t_col} ---")
        
        top_corr = corr_matrix[t_col].drop(target_class_cols, errors="ignore").sort_values(ascending=False)
        
        print("Positivas:")
        print(top_corr.head(10))
        print("-" * 40)
        print("Negativas:")
        print(top_corr.tail(10))

        print("\nVariáveis com correlação mais próxima de 0:")
        near_zero_corr = top_corr.abs().sort_values(ascending=True)
        
        result_df = pd.DataFrame({
            'Correlação Real': top_corr[near_zero_corr.head(10).index],
            'Distância a 0 (|Corr|)': near_zero_corr.head(10)
        })
        print(result_df)
        print("=" * 42)
        
        plt.figure(figsize=(10, 5))
        top_corr.plot(kind="bar", color="cornflowerblue")
    
        
        plt.title(f"Correlação das Variáveis com '{t_col}'")
        plt.ylabel("Coeficiente de Correlação")
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis="y", linestyle="--", alpha=0.7)
        plt.tight_layout()
        plt.show()

### 1.4.1. Atributos Numéricos por Classe

In [ ]:
columns_to_drop = ['age_normalized', 'calcium_baseline', 'systolic_index', 'bmi_level', 'age_level']
df = df.drop(columns=columns_to_drop, errors='ignore')

numeric_cols = [col for col in numeric_cols if col not in columns_to_drop]

for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df_original, x=TARGET_C, y=col)
    plt.title(f"{col} por classe de diagnóstico")
    plt.xlabel("Classe")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

for col in numeric_cols:
    print(f"\n{col} por classe")
    display(df_original.groupby(TARGET_C)[col].agg(
        ["count", "mean", "median", "std", "min", "max"]
    ).round(3))

A análise mostra que apenas um pequeno grupo de atributos numéricos diferencia as classes através da média:

- `HbA1c_level` é o atributo mais discriminante: `DI_HY` tem média de 6.95, enquanto `HY` e `N` ficam ambos em cerca de 5.39. Separa claramente `DI_HY` das restantes, mas não distingue `HY` de `N`.
- `triglycerides` separa `N` (média 209.8) de `HY` e `DI_HY` (média 154-157).
- `age`: a classe `N` é mais jovem e é a única que inclui menores de 18 anos.
- `bmi` tem uma diferença ligeira entre `N` (26.8) e `DI_HY` (28.1), pequena face ao desvio padrão (~7).
- Os restantes atributos têm médias e medianas praticamente iguais nas três classes, pelo que, isoladamente, têm pouco poder discriminativo.

**Heterocedasticidade.** Em vários atributos (`cholesterol`, `systolic_bp`, `diastolic_bp`, `ldl`, `hdl`, `heart_rate`, `salt_intake`, `alcohol_intake`, `sleep_hours`) a classe `N` tem a mesma média das outras, mas um desvio padrão cerca de metade (por exemplo, `ldl`: 16.1 contra 33; `systolic_bp`: 12.1 contra 25). Os desvios padrão de `HY` e `DI_HY` são compatíveis com uma distribuição uniforme em todo o intervalo. Isto significa que as matrizes de covariância diferem entre classes, o que deve ser tido em conta na modelação.

**Outras Notas:** 
- o `ldl_alternative` tem estatísticas globais idênticas às do `ldl`, no entanto, não são iguais, não havendo justificação para as remover.
- Uma vez que o dataset apresenta um valor de outliers ainda considerável, não existe justificação para os remover. Adicionalmente, só conseguiria-mos remover tendo a garantia de que não teriam impacto. Ou seja, apesar da classe `N` ter um número considerável de outliers, não significa que sejam efetivamente "normais", mas sim que, comparados a `DI_HY` ou `HY`, são valores normais. Esta informação, pode justificar, posteriormente a menor capacidade de precisão do poder preditivo dos modelos.


### 1.4.2. Atributos Categóricos por Classe

In [ ]:
# Load the raw dataset specifically for this block so your main dataframe stays encoded
df_raw_analysis = df_original
target = "target_class"

categorical_predictors = [c for c in categorical_cols if c != TARGET_C]

results = []

for col in categorical_predictors:
    if col not in df_raw_analysis.columns or target not in df_raw_analysis.columns:
        print(f"Skipping {col}: required column not found.")
        continue

    # Plot target-class proportions within each category
    proportions = pd.crosstab(
        df_raw_analysis[col],
        df_raw_analysis[target],
        normalize="index"
    )

    proportions.plot(kind="bar", stacked=True, figsize=(8, 4))
    plt.title(f"{col} vs {target}")
    plt.xlabel(col)
    plt.ylabel("Proportion")
    plt.legend(title="Target Class")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


    # Chi-square test of association (using raw counts)
    counts = pd.crosstab(df_raw_analysis[col], df_raw_analysis[target])
    chi2, p_value, dof, _ = chi2_contingency(counts)

    # Calculate max spread across target classes
    proportions_pct = proportions * 100
    max_spread = max(proportions_pct[t].max() - proportions_pct[t].min() for t in proportions_pct.columns)

    results.append({
        "Column": col,
        "Chi2 p-value": p_value,
        "Max target spread (%)": round(max_spread, 1)
    })

display(pd.DataFrame(results).sort_values("Max target spread (%)", ascending=False))

As variáveis age_level, smoking e bmi_level destacam-se pelos valores de p extremamente baixos e pelas maiores diferenças nas proporções das classes (38,8%, 30,0% e 25,8%, respetivamente).

As restantes variáveis, com exceção de gender e high_ldl_cholesterol, também apresentam associações significativas, embora com diferenças menos acentuadas nas proporções das classes. Por outro lado, gender e high_ldl_cholesterol apresentam valores de p maiores que 0,05 e diferenças menores (3,8% e 3,6%), não existindo evidência estatística suficiente de associação com a target_class.

Estes resultados permitem identificar variáveis potencialmente relevantes para a modelação preditiva. Contudo, a significância estatística e as diferenças observadas nas proporções não garantem a capacidade preditiva, pois a contribuição de cada atributo deverá ser avaliada durante a modelação.

## 1.5. Relação entre atributos e `target_regression`

### 1.5.1. Atributos Numéricos

In [ ]:
if TARGET_R in df_graph.columns:
  print(
      "--- Correlação com o Alvo de Regressão"
      f" ({TARGET_R}) ---"
  )
  
  corr_target_r = (
      df_graph.select_dtypes(include="number")
      .corr()[TARGET_R]
      .sort_values(ascending=False)
  )

  print(corr_target_r.head(10))
  print("-" * 40)
  print(corr_target_r.tail(10))
  
  print("\n" + "-" * 50)
  print(f"Variáveis com correlação mais próxima de 0")
  print("-" * 50)
  near_zero_corr = corr_target_r.drop(TARGET_R, errors="ignore").abs().sort_values(ascending=True)
  result_df = pd.DataFrame({
      'Correlação Real': corr_target_r[near_zero_corr.head(10).index],
      'Distância a 0 (|Corr|)': near_zero_corr.head(10)
  })
  print(result_df)

  plt.figure(figsize=(10, 5))
  corr_target_r.drop(TARGET_R, errors="ignore").plot(
      kind="bar", color="cornflowerblue"
  )
  plt.title(f"Correlação das Variáveis com '{TARGET_R}'")
  plt.ylabel("Coeficiente de Correlação")
  plt.xticks(rotation=45, ha="right")
  plt.grid(axis="y", linestyle="--", alpha=0.7)
  plt.tight_layout()
  plt.show()
  
plt.show()

A análise da correlação das variáveis com o alvo da regressão(target_regression) mostra que não há uma alta correlação entre nenhuma das variáveis e o alvo pois os coeficientes de correlação estão muito próximos de zero. As correlações mais positivas incluem crp_level e ldl enquanto as correlações mais negativas estão relacionadas a sleep_hours e diastolic_bp.

Podemos observar nos resultados acima que não há uma relação linear significativa entre as variáveis analisadas e o alvo e portanto pode ser útil testar modelos de aprendizagem automática para relações não lineares. Contudo, é importante frisar que uma correlação baixa não implica necessariamente ausência de uma relação relevante entre as variáveis.

### 1.5.2. Atributos Categóricos

In [ ]:
cat_regression_results = []

for col in categorical_predictors:
    summary = df_original.groupby(col)[TARGET_R].agg(
        ["count", "mean", "median", "std"]
    ).round(3)
    print(f"\n{TARGET_R} por {col}")
    display(summary)

    plt.figure(figsize=(8, 3.5))
    sns.boxplot(data=df_original, x=col, y=TARGET_R)
    plt.title(f"{TARGET_R} por {col}")
    plt.xlabel(col)
    plt.ylabel(TARGET_R)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

Os atributos categóricos não mostram relação relevante com `target_regression`. As médias por categoria variam pouco: entre 132.6 e 135.3 em todas as variáveis, ou seja, diferenças de no máximo ~2.7 unidades, cerca de 7% do desvio padrão do alvo (~37). Os desvios padrão dentro de cada grupo (35 a 39) são muito superiores às diferenças entre grupos, e os boxplots sobrepõem-se quase por completo.

As diferenças ligeiramente maiores surgem em `employment_status` (`Retired` com 135.3 contra 132.8 em `Unemployed`) e `education_level` (`Primary` com 134.8 contra 132.6 em `Tertiary`), em linha com os coeficientes fracos observados em 1.5.1 (`employment_status_Retired` = 0.032 e `education_level_Tertiary` = -0.024), mas sem relevância prática. `gender`, `smoking` e `sugar_consumption` apresentam médias praticamente idênticas entre categorias.

## 1.6. Identificação de Outliers

In [ ]:
outlier_rows = []
for col in numeric_cols + [TARGET_R]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append({
        "variável": col,
        "limite inferior IQR": lower,
        "limite superior IQR": upper,
        "n.º potenciais outliers": int(count),
        "percentagem (%)": round(count / len(df) * 100, 2)
    })

outlier_table = pd.DataFrame(outlier_rows)
display(outlier_table.sort_values("n.º potenciais outliers", ascending=False))

A aplicação do método IQR ajudou a detetar potenciais outliers em várias variáveis, onde as maiores percentagens são heart_rate (18,85%), diastolic_bp (16,45%) e systolic_bp (12,60%) com a maior percentagem de valores irregulares. São também observadas percentagens relevantes em ldl, ldl_alternative e homocysteine_level. Ao contrário destas, variáveis como age, cholesterol e triglycerides não apresentam potenciais outliers segundo este método.

Isto significa que existem valores extremos de certas variáveis que merecem atenção na preparação do conjunto de dados. Mas estes valores podem ser tratados como observações válidas e portanto serão mantidos no trabalho para um análise mais aprofundada.

# 2. Regressão: prever `target_regression`

Nesta secção focamos-nos na construção de dois modelos com o objetivo de prever `target_regression`: uma **Regressão Linear Simples** e uma **Regressão Linear Múltipla**, avaliados com holdout e comparados através de R², MAE e RMSE.

Notas:
- `target_class` **não** é usado como preditor: é um diagnóstico e não um atributo clínico de entrada.
- A seleção do atributo/conjunto de atributos é feita **só com o conjunto de treino** (validação cruzada 5-fold). O conjunto de teste só é usado para a avaliação final, evitando fuga de informação (*data leakage*).
- Como em 1.5 se viu que as correlações com o alvo são muito baixas (|r| ≤ 0.05), incluímos um **modelo de referência** (prever sempre a média do treino): qualquer modelo útil tem de o superar.

## 2.1. Preparação dos dados e holdout

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X = df.drop(columns=['target_class_N', 'target_class_HY', 'target_class_DI_HY', 'target_regression'])

y = df['target_regression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(np.shape(X_train), np.shape(X_test))

## 2.2. Regressão Linear Simples

In [ ]:
results = []

for col in X_train.columns:

    lr = LinearRegression()
    lr.fit(X_train[[col]], y_train)

    y_train_pred = lr.predict(X_train[[col]])

    r2 = r2_score(y_train, y_train_pred)
    rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    mae = mean_absolute_error(y_train, y_train_pred)

    results.append({
        'Atributo': col,
        'R2': r2,
        'RMSE': rmse,
        'MAE': mae
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='R2',
    ascending=False
).reset_index(drop=True)

results_df

In [ ]:
best_feature = results_df.iloc[0]['Atributo']

print("Melhor atributo:", best_feature)
print("R2 de treino:", results_df.iloc[0]['R2'])

In [ ]:
slr = LinearRegression()

slr.fit(
    X_train[[best_feature]],
    y_train
)

y_pred_slr = slr.predict(
    X_test[[best_feature]]
)

In [ ]:
r2_slr = r2_score(y_test, y_pred_slr)
rmse_slr = np.sqrt(mean_squared_error(y_test, y_pred_slr))
mae_slr = mean_absolute_error(y_test, y_pred_slr)

print("Regressão Linear Simples - Resultados")
print("---------------------------------")
print("Feature selecionada:", best_feature)
print("R2:", round(r2_slr, 4))
print("RMSE:", round(rmse_slr, 4))
print("MAE:", round(mae_slr, 4))

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    X_test[best_feature],
    y_test,
    alpha=0.5,
    label='Valores reais'
)

plt.scatter(
    X_test[best_feature],
    y_pred_slr,
    alpha=0.5,
    label='Valores previstos'
)

plt.xlabel(best_feature)
plt.ylabel('target_regression')
plt.title('Regressão Linear Simples - Previsões')
plt.legend()
plt.grid(alpha=0.3)

plt.show()

O modelo da Regressão Linear Simples foi avaliado por cada atributo individualmente, selecionando o atributo com o `R²` mais alto. Dos 32 _features_ testados, o `diastolic_bp` atingiu o maior valor de R² (0.00295), e, por isso, o atributo selecionado para treinar o modelo final.

No entanto, a avaliação no teste demonstrou uma performance preditiva limitada, com o R² em -0.0045, RMSE de 37.3780 e MAE de 31.6304.

O valor negativo indica que o modelo comporta-se ligeiramente pior do que o baseline que prevê o mean no treino dos valores do target. Para além disso, o _scatter plot_ revela a dispersão considerável dos valores reais, enquanto que que os valores previstos concentram-se num pequeno _range_ de valores.

Estes resultados sugerem que o `diastolic_bp` não fornece dados suficientes sozinho para o poder preditivo linear com o `target_regression`. 

No geral, apesar de indetificarmos a melhor feature individual, o modelo de Regressão Linear Simples demonstra-se extremamente limitado quanto a capacidade preditiva.

## 2.3. Regressão Linear Múltipla

In [ ]:
# Regressão Linear Múltipla com todas as features
mlr = LinearRegression()

mlr.fit(X_train, y_train)

y_pred_mlr = mlr.predict(X_test)

r2_mlr = r2_score(y_test, y_pred_mlr)
rmse_mlr = np.sqrt(mean_squared_error(y_test, y_pred_mlr))
mae_mlr = mean_absolute_error(y_test, y_pred_mlr)

print("Regressão Linear Múltipla - Resultados")
print("-----------------------------------")
print("Número de features:", X_train.shape[1])
print("R2:", round(r2_mlr, 4))
print("RMSE:", round(rmse_mlr, 4))
print("MAE:", round(mae_mlr, 4))


In [ ]:
# Baseline

baseline_pred = np.full(
    shape=len(y_test),
    fill_value=y_train.mean()
)

r2_baseline = r2_score(y_test, baseline_pred)
rmse_baseline = np.sqrt(
    mean_squared_error(y_test, baseline_pred)
)
mae_baseline = mean_absolute_error(y_test, baseline_pred)

print("Resultados Base")
print("----------------")
print("R2:", round(r2_baseline, 4))
print("RMSE:", round(rmse_baseline, 4))
print("MAE:", round(mae_baseline, 4))

In [ ]:

from sklearn.feature_selection import SequentialFeatureSelector

# Sequential Feature Selection
sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=4,
    direction='forward',
    scoring='r2',
    cv=5
)

sfs.fit(X_train, y_train)

selected_features_sfs = X_train.columns[sfs.get_support()]

print("Selected Features:")
print(selected_features_sfs.tolist())

In [ ]:

# Multiple Linear Regression - Selected Features

mlr_sfs = LinearRegression()

mlr_sfs.fit(
    X_train[selected_features_sfs],
    y_train
)

y_pred_sfs = mlr_sfs.predict(
    X_test[selected_features_sfs]
)

r2_sfs = r2_score(y_test, y_pred_sfs)
rmse_sfs = np.sqrt(mean_squared_error(y_test, y_pred_sfs))
mae_sfs = mean_absolute_error(y_test, y_pred_sfs)

print("Regressão Linear Múltipla - Resultados SFS")
print("-----------------------------------------")
print("Features Selecionadas:", selected_features_sfs.tolist())
print("R2:", round(r2_sfs, 4))
print("RMSE:", round(rmse_sfs, 4))
print("MAE:", round(mae_sfs, 4))


O modelo de Regressão Linear Múltipla foi treinado utilizando todas os atributos disponíveis para prever `target_regression`.

Este modelo atingiu um R² de -0.0371, umm RMSE de 37.9786 e um MAE de 32.1228 no conjunto de teste. Assim, os resultados indicam uma performance preditiva limitada, uma vez que um R² negativo sugere que o modelo se comporte pior do que a baseline que apenas aplica o _mean_ dos valores target do treino.

## 2.4. Comparação e discussão

In [ ]:

# Final Regression Comparison

regression_results = pd.DataFrame([
    {
        'Model': 'Baseline (Mean)',
        'R2': r2_baseline,
        'RMSE': rmse_baseline,
        'MAE': mae_baseline
    },
    {
        'Model': 'Simple Linear Regression',
        'R2': r2_slr,
        'RMSE': rmse_slr,
        'MAE': mae_slr
    },
    {
        'Model': 'Multiple Linear Regression (All)',
        'R2': r2_mlr,
        'RMSE': rmse_mlr,
        'MAE': mae_mlr
    },
    {
        'Model': 'Multiple Linear Regression (SFS)',
        'R2': r2_sfs,
        'RMSE': rmse_sfs,
        'MAE': mae_sfs
    }
])

regression_results = regression_results.sort_values(
    by='R2',
    ascending=False
).reset_index(drop=True)

regression_results.round(4)


Podemos concluir que nenhum dos métodos revelou que nenhum dos modelos conseguiu ser melhores que o _baseline_. Este, que prevê o valor mean dos valores do target de treino, conseguiu a melhor performance no geral, com um R² de -0.0005, RMSE de 37.3029 e um MAE de 31.6307. 

A Regressão Linear Simples produziu resultados relativemente perto dos do _baseline_, mas não foi melhor na sua capacidade preditiva. Apesar de `distolic_bp`ser selecionado como o melhor atributo individual, a sua contribuição para a previsão dos valores de `target_regression` foi extremamente limitado.

Sobre a Regressão Linear Múltipla, nenhuma das tentativas melhorou os resultados. O modelo utilizou quatro atributos selecionados atraves do Sequencial Feature Selection, que atingiu um R² de -0.0272, enquanto que o modelo com as 32 features objeve um valor R² ainda menor de -0.0371. Ambos os modelos também apresentaram mais erros que a baseline.

Podemos concluir que combinar multiplas features não melhora necessariamente a performance preditiva. Neste caso, reduzir o numero de _features_ pelo SFS também falhou em providenciar melhorias significativas comparada a Regressão Linear Simples.

A consistencia baixa e negativa dos valores de R² indicam que os atributos disponíveis tem um poder explicatório muito baixo quando usamos a regressão linear para prever os valores de `target_regression`. Esta ideia é consistente com a exploração analitica, que identificou correlações lineares muito fracas entre as _features_ individuais e o target.

Assim, baseado-nos nas métricas avaliadas, o _baseline_ mantem-se a técnica mais efetiva de todas das testadas. Estes resultadso demonstram a limitação da regressão linear para esta tarefa preditiva e sugere que os atributos disponiveis não oferecem informação suficiente para previsões precisas através da regressão linear.

# 3. Classificação

## 3.1. Holdout

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

# Start from the original dataset
df_class = df_original.copy()

# Remove target variables and redundant features
columns_to_drop = [
    'target_class',
    'target_regression',
    'age_normalized',
    'calcium_baseline',
    'systolic_index',
    'bmi_level',
    'age_level'
]


X = df_class.drop(columns=columns_to_drop, errors='ignore')
y = df_class['target_class']

# Identify numerical and categorical features
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

# Preprocessing
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore',
                              drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print("Features:", X.shape[1])
print("Classes:")
print(y.value_counts())

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]),

    'LDA': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LinearDiscriminantAnalysis())
    ])
}

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

holdout_results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    holdout_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
        'Macro F1': f1_score(y_test, y_pred, average='macro')
    })

    print(f"\n{name}")
    print(classification_report(y_test, y_pred))

    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        cmap='Blues'
    )
    plt.title(f'Confusion Matrix - {name}')
    plt.show()

holdout_df = pd.DataFrame(holdout_results)
display(holdout_df.round(4))

"""Draft"""

Both Logistic Regression and Linear Discriminant Analysis (LDA) achieved similar classification performance using the Holdout method, with Logistic Regression obtaining slightly higher Accuracy (77.50%) and Macro F1-score (77.33%), compared with LDA (76.75% and 76.86%, respectively).

The `DI_HY` class achieved the highest classification performance in both models, with a recall of approximately 93.7%. In contrast, the `HY` class presented the greatest classification difficulty, frequently being misclassified as `N`. LDA demonstrated better recall for `HY` (62.1% versus 56.0%), while Logistic Regression performed better when identifying the `N` class (81.5% versus 75.7%).

Overall, Logistic Regression demonstrated a marginal advantage in the Holdout experiment. However, the differences between the models were small, and further evaluation using Cross-Validation, LOOCV and Bootstrap is necessary to assess their stability and determine which method is more appropriate for the classification problem.

## 3.2. Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_results = []

for k in [5, 10]:
    cv = StratifiedKFold(
        n_splits=k,
        shuffle=True,
        random_state=42
    )

    for name, model in models.items():

        scores = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring={
                'Accuracy': 'accuracy',
                'Balanced Accuracy': 'balanced_accuracy',
                'Macro F1': 'f1_macro'
            },
            n_jobs=-1
        )

        cv_results.append({
            'Model': name,
            'K': k,
            'Accuracy Mean': scores['test_Accuracy'].mean(),
            'Accuracy Std': scores['test_Accuracy'].std(),
            'Balanced Accuracy Mean': scores['test_Balanced Accuracy'].mean(),
            'Macro F1 Mean': scores['test_Macro F1'].mean(),
            'Macro F1 Std': scores['test_Macro F1'].std()
        })

cv_df = pd.DataFrame(cv_results)
display(cv_df.round(4))

"""Draft"""

Both Logistic Regression and LDA demonstrated similar performance under 5-Fold and 10-Fold Cross-Validation, with Logistic Regression consistently achieving slightly higher average Accuracy and Macro F1-score.

Increasing the number of folds from 5 to 10 did not produce significant changes in average performance, suggesting that both models provide relatively consistent estimates across these configurations.

However, the standard deviation of the fold scores increased for both models when using 10-Fold Cross-Validation. This may be associated with the smaller test sets used in each fold, which can result in greater variability in individual performance scores. Nevertheless, this variability should not be interpreted directly as the variance of the overall Cross-Validation estimator.

Compared with the initial Holdout experiment, both models obtained slightly lower average Accuracy under Cross-Validation, highlighting the importance of evaluating classification performance across multiple data partitions.

Overall, Logistic Regression maintains a marginal advantage over LDA, although further experiments are required to assess the stability of this difference.

## 3.3. Leave-One-Out Cross-Validation (LOOCV)

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
import time

loo = LeaveOneOut()

loocv_results = []

for name, model in models.items():

    start_time = time.time()

    y_pred = cross_val_predict(
        model,
        X,
        y,
        cv=loo,
        n_jobs=-1
    )

    elapsed_time = time.time() - start_time

    loocv_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(y, y_pred),
        'Macro F1': f1_score(y, y_pred, average='macro'),
        'Time (s)': elapsed_time
    })

loocv_df = pd.DataFrame(loocv_results)

display(loocv_df.round(4))

"""Draft"""

Using Leave-One-Out Cross-Validation (LOOCV), Logistic Regression achieved an Accuracy of 75.40%, Balanced Accuracy of 75.74%, and Macro F1-score of 75.78%, outperforming LDA, which obtained 74.30%, 75.04%, and 74.76%, respectively.

These results are consistent with those obtained using 5-Fold and 10-Fold Cross-Validation, reinforcing the slight advantage of Logistic Regression over LDA.

Despite requiring 2,000 training iterations per model, LOOCV demonstrated a manageable computational cost, with execution times of approximately 8.01 seconds for Logistic Regression and 5.77 seconds for LDA.

Although LOOCV uses nearly the entire dataset for training in each iteration, its results alone do not allow a direct estimation of the variance of the resampling procedure. Nevertheless, the similarity between the performance estimates obtained using different Cross-Validation methods suggests that the conclusions are not strongly dependent on the evaluation configuration.

## 3.4. Bootstrap

In [ ]:
from sklearn.base import clone

bootstrap_results = []
n_iterations = 200
rng = np.random.default_rng(42)

for i in range(n_iterations):

    # Bootstrap sample indices
    train_idx = rng.choice(
        len(X),
        size=len(X),
        replace=True
    )

    # Out-of-bag indices
    oob_idx = np.setdiff1d(
        np.arange(len(X)),
        np.unique(train_idx)
    )

    if len(oob_idx) == 0:
        continue

    X_train_boot = X.iloc[train_idx]
    y_train_boot = y.iloc[train_idx]

    X_test_boot = X.iloc[oob_idx]
    y_test_boot = y.iloc[oob_idx]

    for name, model in models.items():

        clf = clone(model)

        clf.fit(X_train_boot, y_train_boot)

        y_pred = clf.predict(X_test_boot)

        bootstrap_results.append({
            'Iteration': i + 1,
            'Model': name,
            'Accuracy': accuracy_score(y_test_boot, y_pred),
            'Balanced Accuracy': balanced_accuracy_score(
                y_test_boot, y_pred
            ),
            'Macro F1': f1_score(
                y_test_boot, y_pred,
                average='macro'
            )
        })

bootstrap_df = pd.DataFrame(bootstrap_results)

bootstrap_summary = bootstrap_df.groupby('Model').agg({
    'Accuracy': ['mean', 'std'],
    'Balanced Accuracy': ['mean', 'std'],
    'Macro F1': ['mean', 'std']
})

display(bootstrap_summary.round(4))

"""Draft""""

Bootstrap evaluation was performed using 200 iterations, with training samples generated through random sampling with replacement and model evaluation conducted on Out-of-Bag (OOB) observations.

Logistic Regression achieved slightly better average performance than LDA, obtaining an Accuracy of 75.01%, Balanced Accuracy of 75.30%, and Macro F1-score of 75.33%, compared with 74.21%, 74.96%, and 74.59% for LDA, respectively.

Both models exhibited relatively low variability across Bootstrap iterations, with Accuracy standard deviations of 0.0150 for Logistic Regression and 0.0143 for LDA. Although LDA demonstrated marginally lower variability in Accuracy, the difference was small.

Overall, the Bootstrap results reinforce the findings obtained using the previous resampling methods, suggesting that Logistic Regression provides a slight performance advantage, while both models demonstrate similar stability across the evaluated samples.

## 3.5. Comparação de Resultados

In [ ]:
n_repeats = 30
repeated_results = []

for seed in range(n_repeats):

    # 1. Repeated Holdout
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    for name, model in models.items():

        clf = clone(model)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        repeated_results.append({
            'Method': 'Holdout',
            'Model': name,
            'Repeat': seed,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Macro F1': f1_score(y_test, y_pred, average='macro')
        })

    # 2. Repeated K-Fold CV
    for k in [5, 10]:

        cv = StratifiedKFold(
            n_splits=k,
            shuffle=True,
            random_state=seed
        )

        for name, model in models.items():

            y_pred = cross_val_predict(
                model,
                X,
                y,
                cv=cv,
                n_jobs=-1
            )

            repeated_results.append({
                'Method': f'{k}-Fold CV',
                'Model': name,
                'Repeat': seed,
                'Accuracy': accuracy_score(y, y_pred),
                'Macro F1': f1_score(y, y_pred, average='macro')
            })

repeated_df = pd.DataFrame(repeated_results)

repeated_summary = (
    repeated_df
    .groupby(['Method', 'Model'])
    .agg(
        Accuracy_Mean=('Accuracy', 'mean'),
        Accuracy_Std=('Accuracy', 'std'),
        Macro_F1_Mean=('Macro F1', 'mean'),
        Macro_F1_Std=('Macro F1', 'std')
    )
    .reset_index()
)

display(repeated_summary.round(4))

In [ ]:
methods_order = ['Holdout', '5-Fold CV', '10-Fold CV']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, group) in zip(axes, repeated_df.groupby('Model')):

    data = [
        group[group['Method'] == method]['Accuracy']
        for method in methods_order
    ]

    ax.boxplot(data, tick_labels=methods_order)

    ax.set_title(name)
    ax.set_ylabel('Accuracy')
    ax.set_xlabel('Resampling Method')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


variance_summary = (
    repeated_df
    .groupby(['Method', 'Model'])
    .agg(
        Accuracy_Mean=('Accuracy', 'mean'),
        Accuracy_Std=('Accuracy', 'std'),
        Accuracy_Variance=('Accuracy', 'var'),
        Macro_F1_Mean=('Macro F1', 'mean'),
        Macro_F1_Std=('Macro F1', 'std')
    )
    .reset_index()
)

display(variance_summary.round(6))



variance_comparison = variance_summary.pivot(
    index='Model',
    columns='Method',
    values='Accuracy_Variance'
)

for method in ['5-Fold CV', '10-Fold CV']:
    variance_comparison[f'{method} Reduction (%)'] = (
        1 - variance_comparison[method] /
        variance_comparison['Holdout']
    ) * 100

display(variance_comparison.round(6))

### Conclusão Geral dos Resultados de Classificação

Neste exercício, foram comparados dois métodos de classificação, a Regressão Logística e a Análise Discriminante Linear (LDA), recorrendo a diferentes técnicas de reamostragem: Holdout, Validação Cruzada com 5 e 10 folds, Leave-One-Out Cross-Validation (LOOCV) e Bootstrap.

De forma geral, a Regressão Logística apresentou resultados ligeiramente superiores aos da LDA, tanto ao nível da Exatidão como da Medida F1 Macro. No entanto, as diferenças observadas foram reduzidas, indicando que ambos os modelos apresentam desempenhos preditivos semelhantes.

A análise das matrizes de confusão obtidas através do Holdout revelou que a classe `DI_HY` foi a mais facilmente identificada por ambos os modelos, enquanto a distinção entre as classes `HY` e `N` apresentou maiores dificuldades. A LDA demonstrou uma maior Sensibilidade (Recall) na identificação da classe `HY`, enquanto a Regressão Logística obteve melhores resultados na classificação da classe `N`.

Relativamente à análise da variância, os resultados das experiências de reamostragem repetida demonstraram que o método Holdout apresentou uma maior variabilidade dos resultados, evidenciando uma maior dependência da divisão dos dados em conjuntos de treino e teste. Por outro lado, a Validação Cruzada com 5 e 10 folds produziu estimativas mais estáveis, tendo a configuração com 10 folds apresentado a menor variância observada ao longo das 30 repetições.

O método Bootstrap também apresentou uma variabilidade relativamente reduzida nas avaliações Out-of-Bag (OOB), embora esta não seja diretamente comparável com a variabilidade obtida através da Validação Cruzada repetida. Por sua vez, o LOOCV apresentou resultados consistentes com os restantes métodos de Validação Cruzada, apesar de não ter sido realizada uma estimativa direta da sua variância.

Com base nos resultados obtidos, **a Regressão Logística foi considerada o método de classificação mais adequado para o conjunto de dados analisado**, uma vez que apresentou um desempenho preditivo global ligeiramente superior de forma consistente nas diferentes técnicas de reamostragem.

Contudo, importa salientar que as diferenças entre os dois modelos foram reduzidas, pelo que seriam necessárias análises estatísticas adicionais e uma avaliação com dados independentes para determinar se a vantagem observada é estatisticamente significativa e generalizável a novos dados.